[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Linear Algebra


## What you will be able to do

Multiply arrays as matrices with `@`, tell a matrix product from an elementwise one, solve a system
of equations with `np.linalg.solve`, and fit a model to the readings with `np.linalg.lstsq` that
recovers their seasonal and daily swings.

## The idea

### The problem

Some calculations combine many values with many weights at once. A blend of three stations, half
Bergen, three tenths Oslo and a fifth Tromso, is a weighted sum for every one of 8,760 hours. Finding
the parabola that passes through three monthly means means solving three equations in three
unknowns. Fitting a curve to a year of readings means choosing a few numbers that make the curve
come as close as possible to 8,760 readings at once.

These can all be written with loops over rows and columns. They are also standard operations of
linear algebra, the mathematics of vectors and matrices, which NumPy hands to BLAS and LAPACK,
long-established libraries written for this work. The operations need their shapes to agree, and the
errors they raise are about shapes.

### What a matrix product is

> The **matrix product** of an array of shape (m, k) and an array of shape (k, n), written `a @ b`,
> is an array of shape (m, n). Each of its values is a row of `a` multiplied by a column of `b`, value
> by value, and added up, so the inner lengths, the two `k`s, must be equal.

### Why it works that way

A weighted sum is a matrix product. Three weights of shape (3,) times the three stations' rows, shape
(3, 8760), give one blended reading for each hour, shape (8760,), which for every column is the
three readings multiplied by their weights and added. `*` does something different. It multiplies values in
the same places, as broadcasting lines them up, and adds nothing.

A system of equations is a matrix product with the unknowns missing. Three equations in three
unknowns are `A @ x = b`, with the coefficients in `A` and the right-hand sides in `b`, and
`np.linalg.solve(A, b)` finds `x`. It fails when two equations say the same thing, because then
there are not enough distinct equations to fix every unknown.

Least squares handles the opposite case, many more equations than unknowns. A curve with five numbers
cannot pass through 8,760 readings, so `np.linalg.lstsq` finds the five that make the sum of squared
misses as small as possible. The curve can be anything built from fixed columns, and a cosine and a
sine of the same period together make a wave that peaks at any time of year, which is what lets a
fit recover when the readings peak as well as by how much.

### Where this shows up

Linear regression in the **scikit-learn, Deep Dive** guide is least squares, and its predictions are
a matrix product of the data with the fitted coefficients. The **SciPy, Deep Dive** guide's optimizers
and its own `linalg` module build on the same operations. A neural network layer is a matrix
product followed by a ufunc.

### What this notebook covers

- a weighted sum as a matrix product with `@`
- `@` against `*`, and the shape rule for `@`
- `np.linalg.solve`, for the parabola through three monthly means
- `np.linalg.lstsq`, for a test line and then a model of a year's readings
- reading the swings and their peaks from the fitted numbers
- several stations fitted in one call

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np

weights = np.array([0.5, 0.5])
pair = np.array([[2.0, 4.0, 6.0],
                 [4.0, 8.0, 0.0]])
print(weights @ pair)
print(np.linalg.solve(np.array([[1.0, 1.0], [1.0, -1.0]]), np.array([10.0, 2.0])))
```

```
[3. 6. 3.]
[6. 4.]
```

The matrix product averaged the two rows, column by column. `np.linalg.solve` found the two numbers
whose sum is 10 and whose difference is 2.

## Setup

Three imports, the year of readings, and a helper that finds an hour's position.

- `numpy`, imported as `np`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings, and `hour_of_year` finds positions

`readings_list` holds four lists of 8,760 readings, one for each station in `STATIONS`, made by
`year_of_readings`, the formula the **Why NumPy** notebook explained, and `readings` is the same
values as one array of shape (4, 8760), with `nan` for Svalbard's silent day. `hour_of_year(7, 1, 12)`
gives the position of noon on 1 July along a station's row. Results from linear algebra are printed
rounded, since their last digits depend on the library that does the arithmetic.

In [1]:
import math
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan


def hour_of_year(month, day, hour=0):
    """The position along a station's row of an hour of 2025. hour_of_year(7, 1, 12) is noon on 1 July."""
    return (datetime(2025, month, day, hour) - START) // timedelta(hours=1)


print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


## Worked examples

### A weighted sum with @

Multiplying the rows of Bergen, Oslo and Tromso by three weights with `@` gives a blended reading
for every hour:

In [2]:
three = readings[[0, 1, 3]]
weights = np.array([0.5, 0.3, 0.2])
blend = weights @ three

print(three.shape, blend.shape)
print(three[:, 0], blend[0].round(2))
print((0.5 * three[0, 0] + 0.3 * three[1, 0] + 0.2 * three[2, 0]).round(2))

(3, 8760) (8760,)
[-3.6 -3.5 -6.7] -4.19
-4.19


The first blended hour, -4.19, is the three first readings times their weights, added up, which the
last line works out by hand. Svalbard is left out because its `nan` hours would make those hours of
the blend `nan`.

### @ against *

In [3]:
left = np.array([[1.0, 2.0], [3.0, 4.0]])
right = np.array([[5.0, 6.0], [7.0, 8.0]])

print(left @ right)
print(left * right)

[[19. 22.]
 [43. 50.]]
[[ 5. 12.]
 [21. 32.]]


In `left @ right`, the top left value is the first row of `left` times the first column of `right`,
added up, 1 × 5 + 2 × 7 = 19. In `left * right` it is the two values in that place multiplied, 1 × 5
= 5.

### The shape rule for @

The inner lengths must match, and they disappear from the result. `.T` transposes an array, swapping
its rows and columns, which also swaps which side of `@` it belongs on:

In [4]:
print(weights.shape, three.shape, (weights @ three).shape)
print(three.T.shape, (three.T @ weights).shape)
print(np.allclose(weights @ three, three.T @ weights))

(3,) (3, 8760) (8760,)
(8760, 3) (8760,)
True


(3,) against (3, 8760) matches on the 3 and leaves (8760,), and so does (8760, 3) against (3,), with
the same values.

### Solving equations with np.linalg.solve

Bergen's monthly means come from the month of every hour, made with `np.repeat` as in the **Fancy
Indexing** notebook:

In [5]:
DAYS_IN_MONTH = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
months = np.repeat(np.arange(12), [24 * length for length in DAYS_IN_MONTH])
monthly = np.array([readings[0][months == month].mean() for month in range(12)])

print(monthly.round(2))

[-0.9   0.12  3.16  7.55 12.06 15.49 16.89 15.86 12.72  8.3   3.8   0.44]


A parabola `c0 + c1 * m + c2 * m ** 2` through January, July and December, months 0, 6 and 11, is
three equations, one for each month, in the three unknown `c`s. Each row of `system` holds 1, `m` and
`m ** 2` for one month:

In [6]:
picked = np.array([0.0, 6.0, 11.0])
system = np.column_stack([np.ones(3), picked, picked ** 2])
targets = monthly[[0, 6, 11]]
parabola = np.linalg.solve(system, targets)

print(system)
print(parabola.round(2))
print(np.allclose(system @ parabola, targets))

[[  1.   0.   0.]
 [  1.   6.  36.]
 [  1.  11. 121.]]
[-0.9   6.37 -0.57]
True


`np.column_stack` made the columns into a 3 by 3 array, and `np.allclose` confirms that the solution
reproduces the three monthly means. The same parabola can predict April, month 3:

In [7]:
april = parabola @ np.array([1.0, 3.0, 3.0 ** 2])

print(april.round(2), monthly[3].round(2))

13.11 7.55


The parabola passes exactly through its three months and misses April by more than five degrees,
because a year of temperatures does not follow a parabola. Three equations fixed three unknowns, and
nothing checked the shape against the other nine months.

### Least squares with np.linalg.lstsq

`np.linalg.lstsq(design, values, rcond=None)` finds the coefficients that bring `design @
coefficients` as close as possible to `values`. On a line with no error in it, `2 + 3x`, it recovers
the line exactly:

In [8]:
x = np.arange(5.0)
line = np.column_stack([np.ones(5), x])

print(np.linalg.lstsq(line, 2 + 3 * x, rcond=None)[0])

[2. 3.]


A model of a year needs a constant for the mean, a cosine and a sine with a period of a year for the
seasons, and a cosine and a sine with a period of a day for the hours. Those are the five columns of
`design`, one row for every hour:

In [9]:
hour_index = np.arange(HOURS)
year_angle = 2 * np.pi * hour_index / HOURS
day_angle = 2 * np.pi * (hour_index % 24) / 24
design = np.column_stack([
    np.ones(HOURS),
    np.cos(year_angle), np.sin(year_angle),
    np.cos(day_angle), np.sin(day_angle),
])

coefficients, squared_misses, rank, singular_values = np.linalg.lstsq(design, readings[0], rcond=None)

print(design.shape, rank)
print(coefficients.round(2))

(8760, 5) 5
[ 8.   -8.63 -2.55 -2.12 -2.12]


`lstsq` returns four things, the coefficients first. A rank of 5 means the five columns are
independent of one another, so the fit has a single best answer. The first coefficient is Bergen's
mean, 8.0. A cosine and a sine of the same period add up to one wave, whose height is
`np.hypot` of their two coefficients and whose peak `np.arctan2` locates:

In [10]:
season_swing = np.hypot(coefficients[1], coefficients[2])
day_swing = np.hypot(coefficients[3], coefficients[4])
season_peak = np.arctan2(coefficients[2], coefficients[1]) % (2 * np.pi) / (2 * np.pi) * HOURS
day_peak = np.arctan2(coefficients[4], coefficients[3]) % (2 * np.pi) / (2 * np.pi) * 24

print(season_swing.round(2), day_swing.round(2))
print(START + timedelta(hours=round(season_peak)))
print(round(day_peak, 2))

9.0 3.0
2025-07-19 04:00:00
15.0


The fit found a seasonal swing of 9.0 degrees peaking at 04:00 on 19 July and a daily swing of 3.0
degrees peaking at 15:00, which are the 9, the 3 and the timing in `year_of_readings` in Setup,
though the fit was given only the readings. What it cannot explain is the wobble:

In [11]:
fitted = design @ coefficients
misses = readings[0] - fitted

print(readings[0, :3], fitted[:3].round(2))
print(misses.std().round(2), np.abs(misses).max().round(2))

[-3.6 -3.7 -3.7] [-2.76 -3.23 -3.54]
0.49 0.85


No reading is more than 0.85 degrees from the fitted curve, which is the formula's wobble of up to
0.8 plus the rounding of every reading to one decimal place.

### Several stations in one call

Given a two-dimensional array of values, `lstsq` fits every column separately with the same design.
`three.T` puts Bergen, Oslo and Tromso in the columns:

In [12]:
fits, *_ = np.linalg.lstsq(design, three.T, rcond=None)

print(fits.shape)
print(fits.round(2))

(5, 3)
[[ 8.    6.5   3.5 ]
 [-8.63 -8.63 -8.63]
 [-2.55 -2.55 -2.55]
 [-2.12 -2.12 -2.12]
 [-2.12 -2.12 -2.12]]


There is one column of coefficients for each station, and only the first row, the means, differs,
because the formula gives every station the same swings at the same times.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/13-linear-algebra-solutions.ipynb).

**1.** Blend Bergen, Oslo and Tromso with the weights 0.2, 0.3 and 0.5 using `@`, and print the blend's first three hours, rounded to two places.

In [13]:
# your code here

**2.** Multiply `[[2, 0], [0, 3]]` and `[[1, 4], [5, 6]]` as arrays, with `@` and with `*`, and print both results.

In [14]:
# your code here

**3.** Fit a straight line to Bergen's monthly means for January to July, months 0 to 6, with `np.linalg.lstsq`, and print the intercept and the slope, rounded to two places.

In [15]:
# your code here

**4.** Fit the model in `design` to Oslo's readings, row 1, and print the mean and the seasonal and daily swings, rounded to two places.

In [16]:
# your code here

**5.** Fit the model to Svalbard's readings, leaving out the missing hours with a mask on both `design` and the readings, and print the date and time at which the fitted seasonal swing peaks.

In [17]:
# your code here

## Common errors

### ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 3 is different from 8760)

In [18]:
three @ weights

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 3 is different from 8760)

The signature is the shape rule written out. Operand 0, `three`, is (n, k) with k = 8760, and operand
1, `weights`, has to start with the same k, and starts with 3. The order of a matrix product matters,
and `weights @ three` or `three.T @ weights` puts the 3s next to each other.

### LinAlgError: Singular matrix

In [19]:
repeated = np.array([0.0, 6.0, 6.0])
same_twice = np.column_stack([np.ones(3), repeated, repeated ** 2])

print(np.linalg.matrix_rank(same_twice), np.linalg.matrix_rank(system))
np.linalg.solve(same_twice, targets)

2 3


LinAlgError: Singular matrix

July went in twice, so two of the three equations are the same, and two distinct equations cannot
fix three unknowns. `np.linalg.matrix_rank` counts the independent rows, 2 here against 3 for the
system that worked. Choose three different months, or, with more equations than unknowns, use
`np.linalg.lstsq`.

### No error, and an elementwise product where a matrix product was meant

In [20]:
print(np.allclose(system * parabola, targets))
print((system * parabola).shape)

False
(3, 3)


Checked with `*`, the solution seems to fail. `system * parabola` broadcast the three coefficients
across every row and multiplied in place, giving a 3 by 3 array instead of three sums, and comparing
that with the targets says `False`. The check needs the matrix product:

In [21]:
print(np.allclose(system @ parabola, targets))

True


### No error, and every coefficient nan

In [22]:
print(np.linalg.lstsq(design, readings[2], rcond=None)[0])

[nan nan nan nan nan]


Svalbard's 24 missing hours went into the sum of squared misses, which made it `nan`, and every
coefficient with it, with no warning. Leave the missing rows out of the design and the values
together, with the same mask:

In [23]:
reported = ~np.isnan(readings[2])
svalbard_fit = np.linalg.lstsq(design[reported], readings[2][reported], rcond=None)[0]

print(reported.sum(), svalbard_fit.round(2))

8736 [-4.5  -8.63 -2.55 -2.12 -2.12]


## Recap

- Write a weighted sum as `weights @ rows`, and keep `*` for multiplying values in place
- Match the inner lengths of a matrix product, and move an array to the other side with `.T`
- Solve as many equations as unknowns with `np.linalg.solve`, and check with `np.allclose(A @ x, b)`
- Fit more equations than unknowns with `np.linalg.lstsq`, after leaving out rows with missing values
- Read a wave's height and peak from a cosine and sine pair with `np.hypot` and `np.arctan2`

## What is next

The **Performance** notebook measures why the array versions in this guide are faster than loops,
how much memory each value takes, and which habits make NumPy code slow again.

---

&#8592; **Previous:** [Random Numbers](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/12-random-numbers.ipynb)  &nbsp;·&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
